In [1]:
import os
import json
import time
import requests
from pathlib import Path
from dotenv import load_dotenv

In [9]:
load_dotenv("../.env", override=True)

ALPHAVANTAGE_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
print(ALPHAVANTAGE_KEY)

PFP2SVUSBI040YNE


In [10]:
COMPANIES = {
    "AFL": "Aflac", "ALL": "Allstate", "AXP": "American Express",
    "BAC": "Bank of America", "BK": "Bank of New York Mellon",
    "BLK": "BlackRock", "C": "Citigroup", "CB": "Chubb",
    "COF": "Capital One", "CRM": "Salesforce", "CSCO": "Cisco",
    "GOOGL": "Alphabet", "HIG": "Hartford Financial",
    "IBM": "IBM", "INTC": "Intel", "MET": "MetLife",
    "META": "Meta", "MS": "Morgan Stanley", "MSFT": "Microsoft",
    "NFLX": "Netflix", "NVDA": "Nvidia", "ORCL": "Oracle",
    "PNC": "PNC Financial", "PRU": "Prudential", "QCOM": "Qualcomm",
    "STT": "State Street", "TFC": "Truist", "TRV": "Travelers",
    "USB": "US Bancorp", "WFC": "Wells Fargo"
}

FINANCIAL_DIR = Path("financial_data")
FINANCIAL_DIR.mkdir(exist_ok=True)

print("Companies:", list(COMPANIES.keys()))
print("Folder ready at:", FINANCIAL_DIR.resolve())

Companies: ['AFL', 'ALL', 'AXP', 'BAC', 'BK', 'BLK', 'C', 'CB', 'COF', 'CRM', 'CSCO', 'GOOGL', 'HIG', 'IBM', 'INTC', 'MET', 'META', 'MS', 'MSFT', 'NFLX', 'NVDA', 'ORCL', 'PNC', 'PRU', 'QCOM', 'STT', 'TFC', 'TRV', 'USB', 'WFC']
Folder ready at: C:\Users\Bikram\earningsiq\data\financial_data


In [4]:
def get_income_statement(ticker):
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "INCOME_STATEMENT",
        "symbol": ticker,
        "apikey": ALPHAVANTAGE_KEY
    }
    
    response = requests.get(url, params=params, timeout=15)
    data = response.json()
    
    if "quarterlyReports" in data:
        return data["quarterlyReports"][:8]
    else:
        return None

In [5]:
def get_earnings_history(ticker):
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "EARNINGS",
        "symbol": ticker,
        "apikey": ALPHAVANTAGE_KEY
    }
    
    response = requests.get(url, params=params, timeout=15)
    data = response.json()
    
    if "quarterlyEarnings" in data:
        return data["quarterlyEarnings"][:8]
    else:
        return None

In [6]:
def save_company_financials(ticker, company_name):
    filename = FINANCIAL_DIR / f"{ticker}_financials.json"
    
    if filename.exists():
        print(company_name, "(" + ticker + ") - Already exists, skipping")
        return
    
    print(company_name, "(" + ticker + "):")
    
    print("  Loading income statement...", end=" ")
    income = get_income_statement(ticker)
    print("OK" if income is not None else "FAILED")
    
    time.sleep(15)
    
    print("  Loading earnings history...", end=" ")
    earnings = get_earnings_history(ticker)
    print("OK" if earnings is not None else "FAILED")
    
    time.sleep(15)
    
    combined = {
        "ticker": ticker,
        "company_name": company_name,
        "income_statement": income,
        "earnings_history": earnings
    }
    
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(combined, f, indent=2)
    
    print("  Saved:", filename.name)

In [11]:
for ticker, company_name in COMPANIES.items():
    save_company_financials(ticker, company_name)
    print()

print("All companies processed.")

Aflac (AFL) - Already exists, skipping

Allstate (ALL) - Already exists, skipping

American Express (AXP) - Already exists, skipping

Bank of America (BAC) - Already exists, skipping

Bank of New York Mellon (BK) - Already exists, skipping

BlackRock (BLK) - Already exists, skipping

Citigroup (C) - Already exists, skipping

Chubb (CB) - Already exists, skipping

Capital One (COF) - Already exists, skipping

Salesforce (CRM) - Already exists, skipping

Cisco (CSCO) - Already exists, skipping

Alphabet (GOOGL) - Already exists, skipping

Hartford Financial (HIG) - Already exists, skipping

IBM (IBM) - Already exists, skipping

Intel (INTC) - Already exists, skipping

MetLife (MET) - Already exists, skipping

Meta (META) - Already exists, skipping

Morgan Stanley (MS) - Already exists, skipping

Microsoft (MSFT) - Already exists, skipping

Netflix (NFLX) - Already exists, skipping

Nvidia (NVDA) - Already exists, skipping

Oracle (ORCL) - Already exists, skipping

PNC Financial (PNC) - A

KeyboardInterrupt: 

In [16]:
print("Verification report:")
print()

for ticker in COMPANIES:
    filename = FINANCIAL_DIR / f"{ticker}_financials.json"
    if filename.exists():
        with open(filename, encoding="utf-8") as f:
            data = json.load(f)
        
        income_count = len(data["income_statement"]) if data["income_statement"] else 0
        earnings_count = len(data["earnings_history"]) if data["earnings_history"] else 0
        
        print(ticker, "- income quarters:", income_count, "| earnings quarters:", earnings_count)

Verification report:

AFL - income quarters: 8 | earnings quarters: 8
ALL - income quarters: 8 | earnings quarters: 8
AXP - income quarters: 8 | earnings quarters: 8
BAC - income quarters: 8 | earnings quarters: 8
BLK - income quarters: 8 | earnings quarters: 8
C - income quarters: 8 | earnings quarters: 8
CB - income quarters: 8 | earnings quarters: 8
COF - income quarters: 8 | earnings quarters: 8
CRM - income quarters: 8 | earnings quarters: 8
CSCO - income quarters: 8 | earnings quarters: 8
GOOGL - income quarters: 8 | earnings quarters: 8
